In [1]:
import warnings
warnings.filterwarnings("ignore")

### Data Preprocessing:

The text dataset is processed and transformed, to use it for model training.
Steps:
- Loading Dataset
- Initialising a tokenizer object for the pretrained model of BERT Transformer
- The transformations on dataset involve
  - Encoding the words with their respective word 'id' from the BERT Model
  - The sentences are padded to make all the sentences of same length in our case the max_length = 200
  - Respective attention vectors are created for each sentence with value '1' for word and value '0' for padded words.
- This transformed requires to have a type of tensorflow tensor data. So we will transform the data into Tensorflow tensor data format.

In [2]:
from datasets import load_dataset 

dataset = load_dataset("ag_news")
split = dataset['train'].train_test_split(test_size=0.1, seed=42)
train_ds = split["train"]
val_ds = split['test']
test_ds = dataset['test']

In [3]:
from transformers import AutoTokenizer

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [4]:
def encode_batch(batch):
    return tokenizer(
        batch['text'],
        max_length = 200,
        truncation = True,
        padding = "max_length"
    )

train_enc = train_ds.map(encode_batch, batched=True)
test_enc = test_ds.map(encode_batch, batched=True)
val_enc = val_ds.map(encode_batch, batched=True)


train_enc = train_enc.rename_column("label", "labels")
test_enc = test_enc.rename_column("label", "labels")
val_enc = val_enc.rename_column("label", "labels")


train_enc.set_format(type = 'tensorflow', columns = ['input_ids','attention_mask', 'labels'])
test_enc.set_format(type = 'tensorflow', columns = ['input_ids','attention_mask', 'labels'])
val_enc.set_format(type = 'tensorflow', columns = ['input_ids','attention_mask', 'labels'])



Map: 100%|██████████████████████████████████████████████████████████████| 12000/12000 [00:04<00:00, 2733.28 examples/s]


In [5]:
import tensorflow as tf

tf_train = train_enc.to_tf_dataset(
    columns=["input_ids", "attention_mask"],
    label_cols=["labels"],
    shuffle=True,
    batch_size=16,
)


tf_val = val_enc.to_tf_dataset(
    columns=["input_ids","attention_mask"],
    label_cols=["labels"],
    shuffle=False,
    batch_size=16,
)

tf_test = test_enc.to_tf_dataset(
    columns=["input_ids", "attention_mask"],
    label_cols=["labels"],
    shuffle=False,
    batch_size=16,
)



### Model Initialization

For the text classification task, we employ the BERT Base Uncased model, which is a pretrained Transformer-based encoder.
BERT is a bidirectional autoencoder architecture trained on large-scale corpora using two objectives:

- Masked Language Modeling (MLM) — learns deep contextual word representations
- Next Sentence Prediction (NSP) — learns relationships between sentences

Because of this pretraining, BERT produces rich semantic embeddings for every token and for the entire sentence through the [CLS] embedding.

#### How BERT is used in our classification task

For sequence (text) classification, we follow the standard fine-tuning setup:

- The input text is tokenized into word-piece tokens and converted to integer IDs.

- These token IDs and their attention masks are passed into BERT’s encoder stack.

- BERT produces a contextual embedding for every token and a special [CLS] embedding, which summarizes the meaning of the entire input sequence.

On top of this [CLS] embedding, we place a fully connected classification layer:

$$
\mathbf{h}_{\text{CLS}} \in \mathbb{R}^{768} \;\longrightarrow\; \text{Dense}(4)
$$


where 4 corresponds to the four AG News categories.

This classification layer is randomly initialized, and only its weights are trained from scratch, while BERT’s encoder weights are fine-tuned during training to adapt the representations to our dataset.

#### What the model learns

- BERT provides context-aware embeddings that capture syntax, semantics, and long-range dependencies.

- The classification head learns to map these embeddings to a 4-class probability distribution using a softmax activation.

- Training optimizes the cross-entropy loss between predicted and true labels.

In [6]:
from transformers import AutoTokenizer, TFBertModel

model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

bert = TFBertModel.from_pretrained(
    model_name,
    from_pt=True  # IMPORTANT: loads PyTorch weights into TF
)


TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.
Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFBertModel: ['cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.decoder.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.bias']
- This IS expected if you are initializing TFBertModel from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertModel from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Al

In [7]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.models import Model

max_len = 200

input_ids = Input(shape=(max_len,), dtype=tf.int32, name="input_ids")
attention_mask = Input(shape=(max_len,), dtype=tf.int32, name="attention_mask")


In [8]:
bert_outputs = bert(
    input_ids=input_ids,
    attention_mask=attention_mask
)

cls_emb = bert_outputs.last_hidden_state[:, 0, :]   # CLS token

x = Dropout(0.2)(cls_emb)
x = Dense(256, activation="relu")(x)
x = Dropout(0.2)(x)
out = Dense(4, activation="softmax")(x)

model = Model(
    inputs=[input_ids, attention_mask],
    outputs=out
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=2e-5),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)


In [9]:
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_ids (InputLayer)         [(None, 200)]        0           []                               
                                                                                                  
 attention_mask (InputLayer)    [(None, 200)]        0           []                               
                                                                                                  
 tf_bert_model (TFBertModel)    TFBaseModelOutputWi  109482240   ['input_ids[0][0]',              
                                thPoolingAndCrossAt               'attention_mask[0][0]']         
                                tentions(last_hidde                                               
                                n_state=(None, 200,                                           

In [10]:
history = model.fit(
    tf_train,
    validation_data = tf_val,
    epochs = 10,
)

Epoch 1/10
 140/6750 [..............................] - ETA: 46:26:11 - loss: 0.5672 - accuracy: 0.7848 

KeyboardInterrupt: 